In [10]:
!pip install -q "transformers>=4.50" accelerate pandas

In [12]:
import os, re, time, unicodedata, zipfile, gc
import torch, pandas as pd
from PIL import Image
from google.colab import userdata
from huggingface_hub import login
from transformers import PaliGemmaProcessor, PaliGemmaForConditionalGeneration

login(token=userdata.get("HF_TOKEN"))

# Correcting the path to the zip file and ensuring extraction if images are missing
if not os.path.exists("veriseti/ana/orijinal"): # Check for a specific representative image folder
    print("Veri seti görüntüleri bulunamadı, arşivden çıkarılıyor...")
    try:
        # Corrected: Use the correct filename 'veriseti.zip'
        with zipfile.ZipFile("/content/veriseti/veriseti.zip", 'r') as zip_ref:
            zip_ref.extractall("veriseti") # Extract contents of zip into the 'veriseti' directory
        print("Veri seti görüntüleri başarıyla çıkarıldı.")
    except FileNotFoundError:
        print("Hata: '/content/veriseti/veriseti.zip' dosyası bulunamadı. Lütfen dosyanın doğru yolda olduğundan emin olun.")
        # You might want to exit or handle this error more gracefully
    except zipfile.BadZipFile:
        print("Hata: '/content/veriseti/veriseti.zip' geçerli bir zip dosyası değil.")
else:
    print("Veri seti görüntüleri zaten mevcut.")

print("GPU:", torch.cuda.get_device_name(0), "| bellek:", round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

Veri seti görüntüleri bulunamadı, arşivden çıkarılıyor...
Veri seti görüntüleri başarıyla çıkarıldı.
GPU: Tesla T4 | bellek: 15.6 GB


In [11]:
IMAGE_SOURCE = "orijinal"          # "orijinal" ya da "letterbox"
GROUPS = ["ana"]                   # ["ana", "taninmis_marka", "kontrol"] ekleyebilirsin
ONLY_VERIFIED = False              # True: sadece manifest'te dogrulandi=x olan satirlar
MAX_NEW_TOKENS = 256

MODELS = {
    224: "google/paligemma2-3b-mix-224",
    448: "google/paligemma2-3b-mix-448",
}
PROMPTS = {
    "ocr":    "ocr",
    "tr_soru": "answer tr Görselde yazan tüm metni yaz.",
    "en_soru": "answer en What is all the text written in the image?",
}
DTYPE = torch.float16 if torch.cuda.get_device_capability(0)[0] < 8 else torch.bfloat16
print("dtype:", DTYPE)

manifest = pd.read_csv("veriseti/manifest.csv", encoding="utf-8-sig").fillna("")
manifest = manifest[manifest["grup"].isin(GROUPS)]
if ONLY_VERIFIED:
    manifest = manifest[manifest["dogrulandi"].str.lower() == "x"]
print(len(manifest), "gorsel")

def image_path(row, res):
    folder = "orijinal" if IMAGE_SOURCE == "orijinal" else str(res)
    # Modified: Added an extra 'veriseti/' to the path due to nested extraction
    return f"veriseti/veriseti/{row['grup']}/{folder}/{row['dosya']}"

dtype: torch.float16
30 gorsel


In [13]:
results = []
for res, model_id in MODELS.items():
    print(f"\n=== {model_id} ===")
    processor = PaliGemmaProcessor.from_pretrained(model_id)
    model = PaliGemmaForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=DTYPE, device_map="auto").eval()
    for _, row in manifest.iterrows():
        current_image_path = image_path(row, res)
        if not os.path.exists(current_image_path):
            print(f"Uyarı: Görüntü dosyası bulunamadı: {current_image_path}. Bu görsel atlanıyor.")
            continue
        image = Image.open(current_image_path).convert("RGB")
        for pname, prompt in PROMPTS.items():
            inputs = processor(text=prompt, images=image, return_tensors="pt").to(DTYPE).to(model.device)
            n_in = inputs["input_ids"].shape[-1]
            t0 = time.time()
            with torch.inference_mode():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            text = processor.decode(out[0][n_in:], skip_special_tokens=True)
            results.append(dict(cozunurluk=res, prompt=pname, dosya=row["dosya"], grup=row["grup"],
                                cikti=text, sure_sn=round(time.time()-t0, 1)))
        print(row["dosya"], "tamam")
        pd.DataFrame(results).to_csv("sonuclar.csv", index=False, encoding="utf-8-sig")
    del model, processor
    gc.collect(); torch.cuda.empty_cache()

res_df = pd.DataFrame(results)
res_df.to_csv("sonuclar.csv", index=False, encoding="utf-8-sig")
res_df.head(6)


=== google/paligemma2-3b-mix-224 ===


preprocessor_config.json:   0%|          | 0.00/424 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/243k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.6MB            

...

[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many i

sahne_33_dikkat_olum_tehlikesi.png tamam


[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.
[transformers] You are passing both `text` and `images` to `PaliGemmaProcessor`. The processor expects special image tokens in the text, as many tokens as there are images per each text. It is recommended to add `<image>` tokens in the very beginning of your text. For this call, we will infer how many images each text has and add special tokens.


sahne_34_kural_kiz_apart.png tamam


,cozunurluk,prompt,dosya,grup,cikti,sure_sn
0,224,ocr,sahne_01_cikartma.png,ana,Onun Hevesl Bitti\nArşın,3.9
1,224,tr_soru,sahne_01_cikartma.png,ana,onun hayesi bitti,0.4
2,224,en_soru,sahne_01_cikartma.png,ana,onun hayesi bitti,0.4
3,224,ocr,sahne_02_terzi_abit_kurt.png,ana,TERZI\nABİT KURT,0.5
4,224,tr_soru,sahne_02_terzi_abit_kurt.png,ana,terzi abit kurt,0.3
5,224,en_soru,sahne_02_terzi_abit_kurt.png,ana,terzi,0.2


In [14]:
pd.set_option("display.max_colwidth", 200)
sample = res_df[res_df["dosya"].isin(manifest["dosya"].head(3))]
sample[["cozunurluk","prompt","dosya","cikti"]]

,cozunurluk,prompt,dosya,cikti
0,224,ocr,sahne_01_cikartma.png,Onun Hevesl Bitti\nArşın
1,224,tr_soru,sahne_01_cikartma.png,onun hayesi bitti
2,224,en_soru,sahne_01_cikartma.png,onun hayesi bitti
3,224,ocr,sahne_02_terzi_abit_kurt.png,TERZI\nABİT KURT
4,224,tr_soru,sahne_02_terzi_abit_kurt.png,terzi abit kurt
5,224,en_soru,sahne_02_terzi_abit_kurt.png,terzi
6,224,ocr,sahne_03_sunar_kayali.png,1967'DEN\nSunar & Kayali\nBER\nZAMANSIZ TASARIM • YÜKSEK KALITE
7,224,tr_soru,sahne_03_sunar_kayali.png,sunar & kayali
8,224,en_soru,sahne_03_sunar_kayali.png,sunar & kayali
90,448,ocr,sahne_01_cikartma.png,Onun Hevesi Bitti\nBeninde Hiçesin


In [15]:
TR_FOLD = str.maketrans("çğıöşüâî", "cgiosuai")

def tr_lower(s):
    return s.replace("İ", "i").replace("I", "ı").lower()

def norm(s):
    s = unicodedata.normalize("NFC", str(s))
    s = tr_lower(s)
    s = re.sub(r"[^\w]+", " ", s, flags=re.UNICODE).replace("_", " ")
    return re.sub(r"\s+", " ", s).strip()

def fold(s):
    return s.translate(TR_FOLD)

def approx_substring_dist(ref, text):
    # ref'in text icindeki en iyi (en az hatali) konumla duzenleme mesafesi
    m, n = len(ref), len(text)
    if m == 0: return 0
    prev = [0] * (n + 1)
    for i in range(1, m + 1):
        cur = [i] + [0] * n
        for j in range(1, n + 1):
            cost = 0 if ref[i-1] == text[j-1] else 1
            cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + cost)
        prev = cur
    return min(prev)

def score(res_df, manifest):
    gt = manifest.set_index("dosya")
    res_df = res_df[res_df["dosya"].isin(gt.index)]   # manifest'ten cikarilan gorselleri atla
    rows = []
    for _, r in res_df.iterrows():
        out_n = norm(r["cikti"]); out_f = fold(out_n)
        for kind in ("ana", "ikincil"):
            for seg in [s.strip() for s in str(gt.loc[r["dosya"], kind + "_metin"]).split("|") if s.strip()]:
                ref = norm(seg)
                if not ref: continue
                rows.append(dict(
                    cozunurluk=r["cozunurluk"], prompt=r["prompt"], dosya=r["dosya"], tur=kind, parca=seg,
                    tam_bulundu=ref in out_n,
                    aksansiz_bulundu=fold(ref) in out_f,
                    hata_orani=round(approx_substring_dist(ref, out_n) / len(ref), 3)))
    return pd.DataFrame(rows)

sk = score(res_df, manifest)
sk.to_csv("skorlar_parca.csv", index=False, encoding="utf-8-sig")
sk.head()

,cozunurluk,prompt,dosya,tur,parca,tam_bulundu,aksansiz_bulundu,hata_orani
0,224,ocr,sahne_01_cikartma.png,ana,Onun Hevesi Bitti,False,False,0.059
1,224,ocr,sahne_01_cikartma.png,ikincil,Benimde Nefesim,False,False,0.667
2,224,tr_soru,sahne_01_cikartma.png,ana,Onun Hevesi Bitti,False,False,0.118
3,224,tr_soru,sahne_01_cikartma.png,ikincil,Benimde Nefesim,False,False,0.667
4,224,en_soru,sahne_01_cikartma.png,ana,Onun Hevesi Bitti,False,False,0.118


In [16]:
ozet = (sk.groupby(["tur", "cozunurluk", "prompt"])
          .agg(parca_sayisi=("parca", "count"),
               tam_bulundu=("tam_bulundu", "mean"),
               aksansiz_bulundu=("aksansiz_bulundu", "mean"),
               ort_hata_orani=("hata_orani", "mean"))
          .round(3).reset_index())
ozet.to_csv("ozet.csv", index=False, encoding="utf-8-sig")
ozet

,tur,cozunurluk,prompt,parca_sayisi,tam_bulundu,aksansiz_bulundu,ort_hata_orani
0,ana,224,en_soru,45,0.111,0.311,0.494
1,ana,224,ocr,45,0.400,0.533,0.159
2,ana,224,tr_soru,45,0.111,0.289,0.503
3,ana,448,en_soru,45,0.267,0.444,0.370
4,ana,448,ocr,45,0.644,0.844,0.087
5,ana,448,tr_soru,45,0.200,0.378,0.428
6,ikincil,224,en_soru,87,0.046,0.057,0.762
7,ikincil,224,ocr,87,0.184,0.230,0.350
8,ikincil,224,tr_soru,87,0.034,0.046,0.751
9,ikincil,448,en_soru,87,0.195,0.264,0.520


In [17]:
ana = sk[(sk["tur"] == "ana") & (sk["prompt"] == "ocr")]
gorsel = ana.pivot_table(index="dosya", columns="cozunurluk", values="tam_bulundu", aggfunc="mean").round(2)
gorsel["fark_448_224"] = gorsel[448] - gorsel[224]
gorsel.sort_values("fark_448_224", ascending=False)

cozunurluk,224,448,fark_448_224
dosya,,,
sahne_01_cikartma.png,0.00,1.00,1.00
sahne_03_sunar_kayali.png,0.00,1.00,1.00
sahne_05_dadas_doner.png,0.00,1.00,1.00
sahne_31_elektrik_pano_plaka.png,0.00,1.00,1.00
sahne_24_ambalaj_yogurt.png,0.00,1.00,1.00
sahne_17_hanturk_anadolu.png,0.00,1.00,1.00
sahne_29_tabelamatik_mockup.png,0.00,1.00,1.00
sahne_33_dikkat_olum_tehlikesi.png,0.00,1.00,1.00
sahne_23_ambalaj_etiket_tarih.png,0.25,0.50,0.25


In [18]:
aksan = sk[(sk["aksansiz_bulundu"]) & (~sk["tam_bulundu"])]
print(len(aksan), "parcada aksan/harf hatasi")
aksan[["cozunurluk","prompt","dosya","parca"]].sort_values("dosya")

72 parcada aksan/harf hatasi


,cozunurluk,prompt,dosya,parca
6,224,ocr,sahne_02_terzi_abit_kurt.png,TERZİ
402,448,ocr,sahne_02_terzi_abit_kurt.png,TERZİ
14,224,ocr,sahne_03_sunar_kayali.png,ZAMANSIZ TASARIM • YÜKSEK KALİTE
15,224,tr_soru,sahne_03_sunar_kayali.png,Sunar & Kayalı
18,224,en_soru,sahne_03_sunar_kayali.png,Sunar & Kayalı
...,...,...,...,...
393,224,tr_soru,sahne_34_kural_kiz_apart.png,KAYITLARIMIZ BAŞLAMIŞTIR
392,224,tr_soru,sahne_34_kural_kiz_apart.png,KURAL KIZ APART
391,224,ocr,sahne_34_kural_kiz_apart.png,KAYITLARIMIZ BAŞLAMIŞTIR
395,224,en_soru,sahne_34_kural_kiz_apart.png,KAYITLARIMIZ BAŞLAMIŞTIR


In [19]:
from google.colab import files
for f in ["sonuclar.csv", "skorlar_parca.csv", "ozet.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>